In [ ]:
%pip install -q qualtran cirq


In [ ]:
import os, sys, json, shutil, numpy as np, subprocess, time, zipfile
from pathlib import Path

COMP = Path('/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation')
WORK = Path('/kaggle/working')
sys.path.insert(0, str(COMP / 'kickmix'))
sys.path.insert(0, str(COMP / 'verifiers'))
sys.path.insert(0, '/kaggle/input/datasets/evanpieser/qcompile-verifiable')
sys.path.insert(0, '/kaggle/input/datasets/evanpieser/qcompile-verifiable/starter_kit')
sys.path.insert(0, '/kaggle/input/datasets/evanpieser/qcompile-verifiable/starter_kit/verifier')

SRC = Path('/kaggle/input/datasets/michaelrustad/vqcc-dfthc-r10-merged-groups/src')
sys.path.insert(0, str(SRC))

# Create dfthcsos package shim in working directory
shim_dir = WORK / 'dfthcsos_shim'
shim_dir.mkdir(exist_ok=True)
(shim_dir / 'dfthcsos').mkdir(exist_ok=True)
(shim_dir / 'dfthcsos' / 'scoreinfra').mkdir(exist_ok=True)

# __init__.py for dfthcsos
(shim_dir / 'dfthcsos' / '__init__.py').write_text('')

# scoreinfra __init__.py
scoreinfra_init = '''
import sys, os
ROOT = "/kaggle/input/datasets/evanpieser/qcompile-verifiable/starter_kit"
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
verifier_path = ROOT + "/verifier"
if verifier_path not in sys.path:
    sys.path.insert(0, verifier_path)
from verifier import dfthc_math as dm
for attr in dir(dm):
    if not attr.startswith("__"):
        globals()[attr] = getattr(dm, attr)
'''
(shim_dir / 'dfthcsos' / 'scoreinfra' / '__init__.py').write_text(scoreinfra_init)

# dfthc_math.py
dfthc_math_content = '''
import sys, os
ROOT = "/kaggle/input/datasets/evanpieser/qcompile-verifiable/starter_kit"
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
verifier_path = ROOT + "/verifier"
if verifier_path not in sys.path:
    sys.path.insert(0, verifier_path)
from verifier import dfthc_math as dm
for attr in dir(dm):
    if not attr.startswith("__"):
        globals()[attr] = getattr(dm, attr)
'''
(shim_dir / 'dfthcsos' / 'scoreinfra' / 'dfthc_math.py').write_text(dfthc_math_content)

# reference.py
reference_content = '''
import sys, os
ROOT = "/kaggle/input/datasets/evanpieser/qcompile-verifiable/starter_kit"
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
verifier_path = ROOT + "/verifier"
if verifier_path not in sys.path:
    sys.path.insert(0, verifier_path)
from verifier import reference as vr
for attr in dir(vr):
    if not attr.startswith("__"):
        globals()[attr] = getattr(vr, attr)
UPPER_ENERGY = -22140.27335
def check_fcidump():
    return vr.fcidump_path("FeMoCo76o113e")
'''
(shim_dir / 'dfthcsos' / 'scoreinfra' / 'reference.py').write_text(reference_content)

# circuits.py
circuits_content = '''
import sys, os
ROOT = "/kaggle/input/datasets/evanpieser/qcompile-verifiable/starter_kit"
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
verifier_path = ROOT + "/verifier"
if verifier_path not in sys.path:
    sys.path.insert(0, verifier_path)
from verifier import circuits as vc
for attr in dir(vc):
    if not attr.startswith("__"):
        globals()[attr] = getattr(vc, attr)
'''
(shim_dir / 'dfthcsos' / 'scoreinfra' / 'circuits.py').write_text(circuits_content)

# costs_prepare.py (empty module to satisfy import)
(shim_dir / 'dfthcsos' / 'scoreinfra' / 'costs_prepare.py').write_text('class SpinFreeData_V1: pass\n')

# Add shim to path
sys.path.insert(0, str(shim_dir))
print('Shim created and loaded')

bp_members = Path('/kaggle/input/datasets/busyaprime/33-qcc-v8-members')
sub_dir = WORK / 'submission'
sub_dir.mkdir(exist_ok=True)
(sub_dir / 'dfthcsos').mkdir(exist_ok=True)

shutil.copy2(bp_members / 'invsqrt.kmb', sub_dir / 'invsqrt.kmb')
shutil.copy2(bp_members / 'dqi.kmb', sub_dir / 'dqi.kmb')
for p in bp_members.glob('dfthcsos__*.kmb'):
    shutil.copy2(p, sub_dir / 'dfthcsos' / p.name.replace('dfthcsos__', ''))

from starter.task1_factorization.solution_io import read_factorization
from starter.task2_angles.encode import TREES, encode_inputs
from verifier.score import load_inputs

bp_npz = bp_members / 'dfthcsos__submission.npz'
params, _ = read_factorization(bp_npz)
inputs = load_inputs('FeMoCo76o113e', params)
npz = np.load(bp_npz, allow_pickle=False)
tree_edges = np.array(npz['tree_edges'])
words = encode_inputs(inputs, tree_edges, 8, 'nearest')

data = {k: np.array(npz[k]) for k in npz.keys()}
data['brot'] = np.int64(8)
data['angle_words'] = words
data['words1'] = words[:76]
data['words2'] = words[76:]

enriched = WORK / 'enriched.npz'
np.savez(enriched, **data)
shutil.copy2(enriched, sub_dir / 'dfthcsos' / 'submission.npz')

print('Setup complete')


In [ ]:
IN = sub_dir / 'dfthcsos' / 'submission.npz'
RUNS = WORK / 'runs'
RUNS.mkdir(exist_ok=True)

print('Step 1: finish.py')
t0 = time.time()
result = subprocess.run(
    [sys.executable, str(SRC / 'finish.py'), str(IN), str(RUNS / 'fin.npz'), '--kappa', '0'],
    capture_output=True, text=True, env=dict(os.environ, PYTHONPATH=str(SRC) + os.pathsep + str(COMP / 'kickmix') + os.pathsep + str(COMP / 'verifiers') + os.pathsep + str(shim_dir) + os.pathsep + '/kaggle/input/datasets/evanpieser/qcompile-verifiable/starter_kit' + os.pathsep + '/kaggle/input/datasets/evanpieser/qcompile-verifiable/starter_kit/verifier')
)
print(result.stdout[-500:] if result.stdout else 'NO STDOUT')
print(result.stderr[-500:] if result.stderr else 'NO STDERR')
print(f'  Time: {time.time()-t0:.0f}s')
print('Output exists:', (RUNS / 'fin.npz').exists())


In [ ]:
print('Step 2: gauge.py')
t0 = time.time()
result = subprocess.run(
    [sys.executable, str(SRC / 'gauge.py'), str(RUNS / 'fin.npz'), str(RUNS / 'g.npz'), '--sweeps', '60'],
    capture_output=True, text=True, env=dict(os.environ, PYTHONPATH=str(SRC) + os.pathsep + str(COMP / 'kickmix') + os.pathsep + str(COMP / 'verifiers') + os.pathsep + str(shim_dir) + os.pathsep + '/kaggle/input/datasets/evanpieser/qcompile-verifiable/starter_kit' + os.pathsep + '/kaggle/input/datasets/evanpieser/qcompile-verifiable/starter_kit/verifier')
)
print(result.stdout[-500:] if result.stdout else 'NO STDOUT')
print(result.stderr[-500:] if result.stderr else 'NO STDERR')
print(f'  Time: {time.time()-t0:.0f}s')
print('Output exists:', (RUNS / 'g.npz').exists())


In [ ]:
from starter.task3_circuits.build_circuits import build_submission

final_npz = RUNS / 'g.npz'
if not final_npz.exists():
    final_npz = RUNS / 'fin.npz'
print('Using factorization:', final_npz)

shutil.copy2(final_npz, sub_dir / 'dfthcsos' / 'submission.npz')

result = build_submission(
    'FeMoCo76o113e', final_npz,
    tree='log_depth', brot=8, batch_size=5, rounding='nearest',
    compiler='qrom',
    output_directory=sub_dir / 'dfthcsos', overwrite=True,
)
print('Build result score:', result['score'])
print('Passes:', result['metrics'] if 'metrics' in result else 'N/A')

import importlib.util
spec = importlib.util.spec_from_file_location('dfthcsos_verify', str(COMP / 'verifiers' / 'dfthcsos' / 'verify.py'))
dfthcsos_verify = importlib.util.module_from_spec(spec)
spec.loader.exec_module(dfthcsos_verify)
CV = dfthcsos_verify.verify
r = CV(sub_dir / 'dfthcsos')
print('Full verify:', r['status'], r['score'])

zip_path = WORK / 'submission_optimized.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for p in sorted(sub_dir.rglob('*')):
        if p.is_file():
            z.write(p, str(p.relative_to(sub_dir)))
print('Wrote', zip_path, zip_path.stat().st_size, 'bytes')
